In [39]:
import requests
import pandas as pd

# Replace 'YOUR_API_KEY' with your actual API key from FMP
API_KEY = "9dfbbfa29d93f4793f246e8fb5ca5e74"
# Exchanges to include
exchanges = ['NYSE', 'NASDAQ', 'AMEX', 'PNK', 'OTC']

# Collect results from all exchanges
all_stocks = []
for exchange in exchanges:
    print(f"Fetching data for {exchange}...")
    url = f"https://financialmodelingprep.com/api/v3/stock-screener?exchange={exchange}&limit=10000&apikey={API_KEY}"
    response = requests.get(url)
    if response.status_code == 200:
        all_stocks.extend(response.json())
    else:
        print(f"Failed to fetch data from {exchange}: {response.status_code}")

# Create DataFrame
df = pd.DataFrame(all_stocks)

# # Optional: Keep only relevant columns
# columns_to_keep = [
#     'symbol', 'companyName', 'exchange', 'sector', 'industry',
#     'marketCap', 'price', 'beta', 'volume', 'isEtf'
# ]
# df = df[columns_to_keep]

# Filter out ETFs if desired
df = df[df['isEtf'] == False].drop(columns='isEtf')
df = df[df['isActivelyTrading'] == True].drop(columns='isActivelyTrading')
df.drop_duplicates("companyName", inplace=True)

# Save to CSV
df.to_csv('us_equities_metadata.csv', index=False)
print("Data saved to 'us_equities_metadata.csv'")



Fetching data for NYSE...
Fetching data for NASDAQ...
Fetching data for AMEX...
Fetching data for PNK...
Fetching data for OTC...
Data saved to 'us_equities_metadata.csv'


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances


def get_peers(ticker, candidate_features=['beta', 'marketCap', 'volume'], top_n=15):
    df = pd.read_csv('us_equities_metadata.csv')
    if ticker not in df['symbol'].values:
        print(f"Ticker {ticker} not found.")
        return pd.DataFrame()

    target = df[df['symbol'] == ticker].iloc[0]
    peers = df[(df['industry'] == target['industry']) & (df['symbol'] != ticker)].copy()

    # Determine usable features (non-null for both target and peers)
    usable_features = []
    for feat in candidate_features:
        if pd.notnull(target.get(feat)) and peers[feat].notnull().sum() > 0:
            usable_features.append(feat)

    if not usable_features:
        print("No sufficient features available for similarity matching.")
        return pd.DataFrame()

    # Drop peers with missing values in usable features
    peers = peers.dropna(subset=usable_features)

    # Include target in matrix for standardization
    temp_df = pd.concat([peers, pd.DataFrame([target])], ignore_index=True)

    # Standardize and compute distances
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(temp_df[usable_features])
    distances = euclidean_distances([X_scaled[-1]], X_scaled[:-1])[0]

    peers['similarity_distance'] = distances
    closest_peers = peers.sort_values('similarity_distance').head(top_n)

    if len(closest_peers) < top_n:
        print(f"Warning: Could only find {len(closest_peers)} peers for {ticker}, which is less than the requested minimum of {top_n}")
    return list(closest_peers['symbol'])

# Example usage
peers = get_peers('AAPL', top_n=15)
print(peers)


['SONY', 'SSNLF', 'XIACY', 'FIS', 'GPRO', 'VUZI', 'SONO', 'TBCH', 'PCRFF', 'KYOCY', 'LPL', 'DBOXF', 'UEIC', 'SHCAF', 'TCLHF']


In [ ]:
# import pandas as pd

# # Load the dataset
# df = pd.read_csv('us_equities_metadata.csv')

# # Function to find peers
# def get_peers(ticker, df, sector=True, industry=True, market_cap_range=0.9):
#     try:
#         company = df[df['symbol'] == ticker].iloc[0]
#     except IndexError:
#         print(f"Ticker {ticker} not found in the dataset.")
#         return pd.DataFrame()

#     peers = df.copy()

#     if sector:
#         peers = peers[peers['sector'] == company['sector']]
#     if industry and 'industry' in df.columns:
#         peers = peers[peers['industry'] == company['industry']]

#     # # Filter by market cap within a certain range
#     # if 'marketCap' in df.columns and not pd.isna(company['marketCap']):
#     #     lower_bound = company['marketCap'] * (1 - market_cap_range)
#     #     upper_bound = company['marketCap'] * (1 + market_cap_range)
#     #     peers = peers[(peers['marketCap'] >= lower_bound) & (peers['marketCap'] <= upper_bound)]

#     # Exclude the company itself
#     peers = peers[peers['symbol'] != ticker]

#     return peers

# # Example usage
# ticker = 'NVDA'
# peer_group = get_peers(ticker, df)
# peer_group.drop_duplicates("companyName", inplace = True)
# print(peer_group[['symbol', 'companyName', 'sector', 'industry', 'marketCap']])

# import os
# import json

# # Directory to store peer groups
# os.makedirs('peer_groups', exist_ok=True)

# for ticker in df['symbol'].unique():
#     peers = get_peers(ticker, df)
#     peers_list = peers['symbol'].tolist()
#     with open(f'peer_groups/{ticker}_peers.json', 'w') as f:
#         json.dump(peers_list, f)

# import json

# def load_peers(ticker):
#     try:
#         with open(f'peer_groups/{ticker}_peers.json', 'r') as f:
#             peers = json.load(f)
#         return peers
#     except FileNotFoundError:
#         print(f"No peer group found for {ticker}.")
#         return []

# # Example usage
# ticker = 'AAPL'
# peers = load_peers(ticker)
# print(f"Peers for {ticker}: {peers}")

      symbol                                        companyName      sector  \
1        TSM  Taiwan Semiconductor Manufacturing Company Lim...  Technology   
123      UMC                United Microelectronics Corporation  Technology   
283      ASX                   ASE Technology Holding Co., Ltd.  Technology   
440      STM                            STMicroelectronics N.V.  Technology   
1054    ONTO                               Onto Innovation Inc.  Technology   
...      ...                                                ...         ...   
21765   NMGC                               NeoMagic Corporation  Technology   
21902  SPVNF                         Spectra7 Microsystems Inc.  Technology   
21978   STRI                                 STR Holdings, Inc.  Technology   
22420   CXCQ                                        Cardxx Inc.  Technology   
23074  MMATQ                                Meta Materials Inc.  Technology   

             industry     marketCap  
1      Semico